## Class Creation - Data Puller

In [1]:
# CLASS CREATION - STOCK_DATA_PULLER

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

class AlpacaDataPuller:
    """
    Pulls Stock Data from Alpaca API and establishes quartile-based baseline targets.
    """
    def __init__(
        self, 
        symbols: list[str], 
        api_key: str,
        secret_key: str,
        timeframe_amount: int = 5,
        timeframe_unit: str = "minute"
    ):
        self.symbols = symbols
        self.api_key = api_key
        self.secret_key = secret_key
        self.timeframe_amount = timeframe_amount
        self.timeframe_unit = timeframe_unit
        
        # Dictionary to store pulled self.dfs for different timeframes
        self.data = {}

        # Initialize Alpaca Client
        self.client = StockHistoricalDataClient(api_key, secret_key)

    def get_timeframe(self, amount=None, unit=None):
        if amount is None:
            amount = self.timeframe_amount
        if unit is None:
            unit = self.timeframe_unit

        unit = unit.lower()
        unit_map = {
            "minute": TimeFrameUnit.Minute,
            "min": TimeFrameUnit.Minute,
            "hour": TimeFrameUnit.Hour,
            "day": TimeFrameUnit.Day,
            "week": TimeFrameUnit.Week,
        }

        if unit not in unit_map:
            raise ValueError("timeframe_unit must be: minute, hour, day, or week")

        return TimeFrame(amount, unit_map[unit])

    def pull_symbol_data(self, symbol, start=None, end=None, timeframe_amount=None, timeframe_unit=None):
        if end is None:
            end_date = datetime.today()
        else:
            end_date = datetime(*end)

        if start is None:
            start_date = end_date - timedelta(days=365 * 10)
        else:
            start_date = datetime(*start)

        request = StockBarsRequest(
            symbol_or_symbols=symbol,
            timeframe=self.get_timeframe(timeframe_amount, timeframe_unit),
            start=start_date,
            end=end_date,
            feed=DataFeed.IEX  # Use IEX for free/paper subscriptions
        )

        bars = self.client.get_stock_bars(request)
        return bars.df

    def pull_symbols_data(self, symbols=None, start=None, end=None, timeframe_amount=None, timeframe_unit=None):
        if symbols is None:
            symbols = self.symbols

        data = {}
        for symbol in symbols:
            data[symbol] = self.pull_symbol_data(symbol, start, end, timeframe_amount, timeframe_unit)
        return data

    def pull_multi_timeframe_data(self, timeframes: dict, start=None, end=None):
        """
        Pulls data for multiple timeframes and stores it in self.data.
        timeframes dictionary format: {'daily': (1, 'day'), 'weekly': (1, 'week')}
        """
        for label, (amount, unit) in timeframes.items():
            print(f"Pulling data for timeframe: '{label}' ({amount} {unit})...")
            self.data[label] = self.pull_symbols_data(
                symbols=self.symbols,
                start=start,
                end=end,
                timeframe_amount=amount,
                timeframe_unit=unit
            )
        print("All timeframes pulled successfully!")

    # --- 5-CLASS QUARTILE BASELINE LOGIC ---

    def classify_return(self, r, q1, q3, thresh):
        if r < q1:
            return 0  # Strong Down
        elif r < -thresh:
            return 1  # Moderate Down
        elif r <= thresh:
            return 2  # Flat
        elif r <= q3:
            return 3  # Moderate Up
        else:
            return 4  # Strong Up

    def create_baselines(self, timeframe_label, zero_thresh=0.02, close_col='close'):
        """
        Establishes 5 classes based on quartiles and a zero threshold.
        Appends the 'Target' column to each symbol's DataFrame in self.data[timeframe_label].
        """
        if timeframe_label not in self.data:
            raise ValueError(f"Timeframe '{timeframe_label}' data has not been pulled yet.")
            
        print(f"\n=== 5-Class Baselines for Timeframe: {timeframe_label.upper()} ===")
        
        class_names = {
            0: "Strong Down (R < Q1)",
            1: "Moderate Down (Q1 <= R < -thresh)",
            2: "Flat (-thresh <= R <= thresh)",
            3: "Moderate Up (thresh < R <= Q3)",
            4: "Strong Up (R > Q3)"
        }
        
        for symbol, df in self.data[timeframe_label].items():
            df = df.copy()
            
            # Calculate next period's percentage return grouped by symbol if combined,
            # or just simple return if it is a single-ticker dataframe.
            if 'symbol' in df.columns:
                df['Next_Return'] = df.groupby('symbol')[close_col].transform(lambda x: x.pct_change().shift(-1))
            elif isinstance(df.index, pd.MultiIndex) and 'symbol' in df.index.names:
                df['Next_Return'] = df.groupby(level='symbol')[close_col].transform(lambda x: x.pct_change().shift(-1))
            else:
                df['Next_Return'] = df[close_col].pct_change().shift(-1)
            
            # Clean returns for calculating statistics
            clean_returns = df['Next_Return'].dropna()
            
            # Group and calculate targets per symbol to prevent target bleeding
            if 'symbol' in df.columns or (isinstance(df.index, pd.MultiIndex) and 'symbol' in df.index.names):
                def get_symbol_target(group):
                    g_clean = group['Next_Return'].dropna()
                    if len(g_clean) == 0:
                        group['Target'] = np.nan
                        return group
                    g_q1 = g_clean.quantile(0.25)
                    g_q3 = g_clean.quantile(0.75)
                    g_avg_mov = g_clean.abs().mean()
                    g_thresh = zero_thresh * g_avg_mov
                    
                    group['Target'] = group['Next_Return'].apply(
                        lambda x: self.classify_return(x, g_q1, g_q3, g_thresh) if pd.notna(x) else np.nan
                    )
                    return group
                
                if 'symbol' in df.columns:
                    df = df.groupby('symbol', group_keys=False).apply(get_symbol_target)
                else:
                    df = df.groupby(level='symbol', group_keys=False).apply(get_symbol_target)
            else:
                q1 = clean_returns.quantile(0.25)
                q3 = clean_returns.quantile(0.75)
                avg_movement = clean_returns.abs().mean()
                thresh = zero_thresh * avg_movement
                df['Target'] = df['Next_Return'].apply(
                    lambda x: self.classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
                )
            
            # Drop the last row which contains NaN target
            df = df.dropna(subset=['Target'])
            df['Target'] = df['Target'].astype(int)
            
            # Store the updated DataFrame back in self.data
            self.data[timeframe_label][symbol] = df
            
            # Print class distribution per symbol
            if 'symbol' in df.columns:
                symbols_present = df['symbol'].unique()
            elif isinstance(df.index, pd.MultiIndex) and 'symbol' in df.index.names:
                symbols_present = df.index.get_level_values('symbol').unique()
            else:
                symbols_present = [symbol]
                
            for sym in symbols_present:
                if 'symbol' in df.columns:
                    sym_df = df[df['symbol'] == sym]
                elif isinstance(df.index, pd.MultiIndex) and 'symbol' in df.index.names:
                    sym_df = df.xs(sym, level='symbol', drop_level=False)
                else:
                    sym_df = df
                    
                sym_clean_returns = sym_df['Next_Return'].dropna()
                sym_q1 = sym_clean_returns.quantile(0.25)
                sym_q3 = sym_clean_returns.quantile(0.75)
                sym_avg = sym_clean_returns.abs().mean()
                sym_thresh = zero_thresh * sym_avg
                
                print(f"\nSymbol: {sym}")
                print(f"  Average Absolute Movement: {sym_avg * 100:.4f}%")
                print(f"  Flat Threshold (+/- {zero_thresh*100}% of Avg):  {sym_thresh * 100:.4f}%")
                print(f"  Q1 (25th percentile):             {sym_q1 * 100:.4f}%")
                print(f"  Q3 (75th percentile):             {sym_q3 * 100:.4f}%")
                
                counts = sym_df['Target'].value_counts().sort_index()
                total = len(sym_df)
                for c, count in counts.items():
                    print(f"  Class {c} | {class_names[c]:<35} : {count:<5} ({count/total*100:.2f}%)")


## SPY Example using AlpacaDataPuller

In [2]:
## TEST CELL

# REQUIRED INPUTS
API_KEY = "PKEN63LDIFPD43FZN5JQXXXHXV"
SECRET_KEY = "3pmEQxuC9R5vGL4y6YSfDMWq9fgSM8nBZae8QGHyoE3C"
SYMBOLS = ['SPY']

# PULL SPY HISTORICALS INTO DICTIONARY BY TIME UNIT

puller_hrly = AlpacaDataPuller(SYMBOLS,API_KEY,SECRET_KEY, timeframe_amount=1, timeframe_unit="hour")

puller_daily = AlpacaDataPuller(SYMBOLS,API_KEY,SECRET_KEY, timeframe_amount=1, timeframe_unit="day")

puller_weekly = AlpacaDataPuller(SYMBOLS,API_KEY,SECRET_KEY, timeframe_amount=1, timeframe_unit="week")

SPY_dict = {
    'hourly': puller_hrly.pull_symbols_data(),
    'daily': puller_daily.pull_symbols_data(),
    'weekly': puller_weekly.pull_symbols_data()
}

In [2]:
# Pull data for 3 representative companies and Index (SPY,MU, TSLA, GOOG)

MULTI_SYMBOL = ["MU", "GOOG", "TSLA", "SPY"]

API_KEY = "PKEN63LDIFPD43FZN5JQXXXHXV"
SECRET_KEY = "3pmEQxuC9R5vGL4y6YSfDMWq9fgSM8nBZae8QGHyoE3C"

# Initialize puller
puller = AlpacaDataPuller(MULTI_SYMBOL, API_KEY, SECRET_KEY)

# Define configurations to download (e.g. 1 hour, 1 day, 1 week bars)
timeframe_configs = {
    'hourly': (1, 'hour'),
    'daily': (1, 'day'),
    'weekly': (1, 'week')
}

# Pull 10 years of data (start dates will be limited by Alpaca for hourly)
puller.pull_multi_timeframe_data(timeframe_configs, start=(2016, 6, 20))

# Compute targets & prints metric summaries
puller.create_baselines('hourly')
puller.create_baselines('daily')
puller.create_baselines('weekly')


Pulling data for timeframe: 'hourly' (1 hour)...
Pulling data for timeframe: 'daily' (1 day)...
Pulling data for timeframe: 'weekly' (1 week)...
All timeframes pulled successfully!

=== 5-Class Baselines for Timeframe: HOURLY ===

Symbol: MU
  Average Absolute Movement: 0.7475%
  Flat Threshold (+/- 2.0% of Avg):  0.0149%
  Q1 (25th percentile):             -0.4470%
  Q3 (75th percentile):             0.4928%
  Class 0 | Strong Down (R < Q1)                : 2680  (25.00%)
  Class 1 | Moderate Down (Q1 <= R < -thresh)   : 2440  (22.76%)
  Class 2 | Flat (-thresh <= R <= thresh)       : 208   (1.94%)
  Class 3 | Moderate Up (thresh < R <= Q3)      : 2712  (25.30%)
  Class 4 | Strong Up (R > Q3)                  : 2680  (25.00%)

Symbol: GOOG
  Average Absolute Movement: 0.4732%
  Flat Threshold (+/- 2.0% of Avg):  0.0095%
  Q1 (25th percentile):             -0.2901%
  Q3 (75th percentile):             0.3130%
  Class 0 | Strong Down (R < Q1)                : 2644  (25.00%)
  Class 1 | M


    Different timeframes represent very different data distributions and market microstructures:
1. **The Volatility vs. Noise Trade-off**
    - **Hourly Data (High Noise, High Volume):**
        - Characteristics: You have a huge number of data points (44k+ rows), which ML models love. However, the price moves are very small (average absolute move is only 0.58%) and dominated by short-term noise.
        - ML Hypothesis: Simpler models (like Logistic Regression) might perform better here because they resist overfitting to noise, whereas complex tree models (like XGBoost) might overfit unless heavily regularized.
    
    - **Weekly Data (Low Noise, Low Volume): Characteristics: You have very clean signals and larger price moves (average absolute move is 4.74%), but very few data points (only ~500 rows).**
        - ML Hypothesis: Tree models might struggle here due to the lack of training data, but if they work, the trades are highly profitable.

    - **Daily Data (The Benchmark):**
        - Characteristics: The middle ground (average absolute move of 1.95%), with about 2,500 data points.

2. **By training separate models for the three timeframes, you can answer the following questions:**
    - **Which timeframe is the most "predictable"?**
        - Compare the test set classification accuracy of each model against its respective baseline (e.g. comparing the model's daily accuracy to the 25% random quartile baseline).

    - **Which model handles noise best?**
        - Does LightGBM/XGBoost outperform Logistic Regression on hourly data (where it can learn complex patterns from 44,000 rows) or on daily data?

    - **How do transaction costs impact performance?**
        - An hourly model might have 60% accuracy, but if it recommends trading 10 times a day, the broker fees and slippage will wipe out all profits. A weekly model trades less frequently and is much cheaper to execute in real life. You can show this comparison!

In [3]:
import pandas as pd

'''create simple dataframe tables
symbol: Multiindex
'''
# Concatenate individual DataFrames since puller now returns dictionary with separate symbol keys
hourly_data = pd.concat(puller.data['hourly'].values())
weekly_data = pd.concat(puller.data['weekly'].values())
daily_data  = pd.concat(puller.data['daily'].values())


In [4]:
hourly_data.reset_index(inplace=True)
weekly_data.reset_index(inplace=True)
daily_data.reset_index(inplace=True)

### Feature Engineering: The Weekly Warmup & Dynamic Lookback Problem

When using technical indicators in time-series data, we lose the first $N$ rows as a "warmup period" (e.g. a 200-period average requires 200 days/weeks of history before it can output its first value). 

* **Hourly / Daily**: 200 periods is highly feasible (only 10 months of history on daily, and less than 30 trading days on hourly). We only lose ~8% of daily rows.
* **Weekly**: 200 periods is **200 weeks (~4 years)** of data. With a 10-year dataset, using a 200-week lookback forces us to discard **40%** of our training rows.

#### Dynamic Lookback Strategy:
To solve this, our updated `feature_create` class accepts a `timeframe` argument in its constructor and dynamically scales down the rolling windows for weekly data (capping lookbacks at 30 weeks). This preserves valuable data while keeping the indicators stationary and representative.

In [5]:
class feature_create:
    '''Class to help incorporate all methods for 
    metric creation. Computes metrics grouped by symbol to prevent data leakage
    and dynamically scales lookback windows depending on the timeframe.'''

    def __init__(self, df, timeframe="daily"):
        self.df = df
        self.timeframe = timeframe.lower()
    
    def calculate_moving_avg(self):
        # Set windows based on timeframe (weekly is scaled down to prevent massive warmup data loss)
        if self.timeframe == "weekly":
            windows = (5, 10, 20, 30)
        else:
            windows = (10, 20, 50, 100, 200)
            
        # Group by symbol before rolling or ewm calculations to prevent cross-symbol bleeding
        for w in windows:
            self.df[f'SMA_{w}'] = self.df.groupby('symbol')['close'].transform(lambda x: x.rolling(window=w).mean())
            self.df[f'ewm_{w}'] = self.df.groupby('symbol')['close'].transform(lambda x: x.ewm(span=w, adjust=False).mean())
            self.df[f'price_to_ma{w}'] = self.df['close'] / self.df[f'SMA_{w}'] - 1
            
        # Moving average crossover
        if self.timeframe == "weekly":
            self.df['ma_cross'] = self.df['SMA_10'] / self.df['SMA_30'] - 1
        else:
            self.df['ma_cross'] = self.df['SMA_50'] / self.df['SMA_200'] - 1
        return self

    def bollinger(self):
        self.df['BB_Mid'] = self.df.groupby('symbol')['close'].transform(lambda x: x.rolling(window=20).mean())
        bb_std = self.df.groupby('symbol')['close'].transform(lambda x: x.rolling(window=20).std())
        self.df['BB_Upper'] = self.df['BB_Mid'] + 2 * bb_std
        self.df['BB_Lower'] = self.df['BB_Mid'] - 2 * bb_std
        
        # Bollinger Band Percentile (bb_pct - where price sits in the band: 0 = bottom, 1 = top)
        denom = self.df['BB_Upper'] - self.df['BB_Lower']
        self.df['bb_pct'] = np.where(denom == 0, 0.5, (self.df['close'] - self.df['BB_Lower']) / denom)
        return self

    def RSI(self):
        def compute_rsi(series):
            delta = series.diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)
            avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            rs = avg_gain / avg_loss
            return 100 - (100 / (1 + rs))
            
        self.df['RSI'] = self.df.groupby('symbol')['close'].transform(compute_rsi)
        return self

    def MACD(self):
        def compute_macd_line(series):
            return series.ewm(span=12, adjust=False).mean() - series.ewm(span=26, adjust=False).mean()
        def compute_macd_signal(macd_series):
            return macd_series.ewm(span=9, adjust=False).mean()
            
        # Divided by close to keep MACD stationary over long horizons
        macd_line = self.df.groupby('symbol')['close'].transform(compute_macd_line)
        self.df['MACD'] = macd_line / self.df['close']
        self.df['MACD_Signal'] = self.df.groupby('symbol')['MACD'].transform(compute_macd_signal)
        self.df['MACD_Hist'] = self.df['MACD'] - self.df['MACD_Signal']
        return self

    def ret(self):
        def ret_1(series):
            return series.pct_change()
        def log_ret(series):
            return np.log(series).diff()

        # daily returns (the core signal)
        self.df['ret_1'] = self.df.groupby('symbol')['close'].transform(ret_1)
        self.df['log_ret'] = self.df.groupby('symbol')['close'].transform(log_ret)
        
        # yesterday / 2 days ago / 3 days ago returns (short-term memory)
        for lag in (1, 2, 3):
            self.df[f"ret_lag{lag}"] = self.df.groupby('symbol')["ret_1"].transform(lambda x: x.shift(lag))
        return self

    def momentum(self):
        if self.timeframe == "weekly":
            horizons = (2, 4, 8, 12, 26)  # 2 weeks, 1 month, 2 months, 3 months, 6 months
        else:
            horizons = (10, 20, 30, 50, 100, 200)
            
        for k in horizons:
            self.df[f"mom_{k}"] = self.df.groupby('symbol')['close'].transform(lambda x: x / x.shift(k) - 1)
        return self

    def volatility_and_volume(self):
        # Realized volatility (rolling standard deviation of returns)
        vol_w = 10 if self.timeframe == "weekly" else 20
        self.df[f'volatility_{vol_w}'] = self.df.groupby('symbol')['ret_1'].transform(lambda x: x.rolling(window=vol_w).std())
        
        # Volume ratios
        vol_windows = (4, 10, 26) if self.timeframe == "weekly" else (10, 20, 50)
        for w in vol_windows:
            vol_avg = self.df.groupby('symbol')['volume'].transform(lambda x: x.rolling(window=w).mean())
            self.df[f'vol_ratio{w}'] = self.df['volume'] / vol_avg
        return self

    def advanced_indicators(self):
        # 1. Trading Range
        self.df['range_hl'] = (self.df['high'] - self.df['low']) / self.df['close']
        
        # 2. Overnight Gap
        prev_close = self.df.groupby('symbol')['close'].shift(1)
        self.df['gap'] = (self.df['open'] - prev_close) / prev_close
        
        # 3. Average True Range
        tr1 = self.df['high'] - self.df['low']
        tr2 = (self.df['high'] - prev_close).abs()
        tr3 = (self.df['low'] - prev_close).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        
        alpha = 1/10 if self.timeframe == "weekly" else 1/14
        min_periods = 10 if self.timeframe == "weekly" else 14
        atr_raw = tr.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        self.df[f'atr_{int(1/alpha)}'] = atr_raw / self.df['close']
        
        # 4. Average Directional Index
        up_move = self.df.groupby('symbol')['high'].diff()
        down_move = -self.df.groupby('symbol')['low'].diff()
        plus_dm = pd.Series(np.where((up_move > down_move) & (up_move > 0), up_move, 0.0), index=self.df.index)
        minus_dm = pd.Series(np.where((down_move > up_move) & (down_move > 0), down_move, 0.0), index=self.df.index)
        
        plus_dm_smoothed = plus_dm.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        minus_dm_smoothed = minus_dm.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        
        plus_di = 100 * plus_dm_smoothed / atr_raw
        minus_di = 100 * minus_dm_smoothed / atr_raw
        dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
        self.df[f'adx_{int(1/alpha)}'] = dx.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        
        # 5. Money Flow Index
        typical = (self.df['high'] + self.df['low'] + self.df['close']) / 3
        money_flow = typical * self.df['volume']
        typical_shift = typical.groupby(self.df['symbol']).shift(1)
        
        pos_flow = pd.Series(np.where(typical > typical_shift, money_flow, 0.0), index=self.df.index)
        neg_flow = pd.Series(np.where(typical < typical_shift, money_flow, 0.0), index=self.df.index)
        
        mfi_w = 10 if self.timeframe == "weekly" else 14
        pos_mf = pos_flow.groupby(self.df['symbol']).transform(lambda x: x.rolling(mfi_w).sum())
        neg_mf = neg_flow.groupby(self.df['symbol']).transform(lambda x: x.rolling(mfi_w).sum())
        self.df[f'mfi_{mfi_w}'] = 100 - (100 / (1 + pos_mf / neg_mf))
        
        # 6. Chaikin Money Flow
        denom = self.df['high'] - self.df['low']
        denom = np.where(denom == 0, 1e-9, denom)
        mfm = ((self.df['close'] - self.df['low']) - (self.df['high'] - self.df['close'])) / denom
        mfv = mfm * self.df['volume']
        
        cmf_w = 10 if self.timeframe == "weekly" else 20
        num_sum = mfv.groupby(self.df['symbol']).transform(lambda x: x.rolling(cmf_w).sum())
        denom_sum = self.df['volume'].groupby(self.df['symbol']).transform(lambda x: x.rolling(cmf_w).sum())
        self.df[f'cmf_{cmf_w}'] = num_sum / denom_sum
        return self

    def create_all_features(self):
        '''Runs all feature engineering methods on the DataFrame in one call.'''
        self.calculate_moving_avg()
        self.bollinger()
        self.RSI()
        self.MACD()
        self.ret()
        self.momentum()
        self.volatility_and_volume()
        self.advanced_indicators()
        return self


In [6]:
hourly_data_feature = feature_create(hourly_data, timeframe="hourly").create_all_features()
daily_data_feature = feature_create(daily_data, timeframe="daily").create_all_features()
weekly_data_feature = feature_create(weekly_data, timeframe="weekly").create_all_features()

# View one of the updated DataFrames with all columns!
daily_data_feature.df.head()


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,Next_Return,...,volatility_20,vol_ratio10,vol_ratio20,vol_ratio50,range_hl,gap,atr_14,adx_14,mfi_14,cmf_20
0,MU,2020-07-27 04:00:00+00:00,50.670,51.700,50.495,51.605,166507.0,1860.0,51.367901,-0.028389,...,NaN,NaN,NaN,NaN,0.023350,NaN,NaN,NaN,NaN,NaN
1,MU,2020-07-28 04:00:00+00:00,51.200,51.225,50.050,50.140,175657.0,1619.0,50.429399,0.004188,...,NaN,NaN,NaN,NaN,0.023434,-0.007848,NaN,NaN,NaN,NaN
2,MU,2020-07-29 04:00:00+00:00,50.265,50.615,49.805,50.350,163859.0,1823.0,50.204913,0.007349,...,NaN,NaN,NaN,NaN,0.016087,0.002493,NaN,NaN,NaN,NaN
3,MU,2020-07-30 04:00:00+00:00,49.750,50.745,49.110,50.720,313145.0,2201.0,49.813198,-0.014294,...,NaN,NaN,NaN,NaN,0.032236,-0.011917,NaN,NaN,NaN,NaN
4,MU,2020-07-31 04:00:00+00:00,50.650,50.670,49.270,49.995,277686.0,2331.0,49.930770,0.008001,...,NaN,NaN,NaN,NaN,0.028003,-0.001380,NaN,NaN,NaN,NaN


In [7]:
# 3. View number of NaN values per column for hourly data
hourly_data_feature.df.isna().sum()

symbol              0
timestamp           0
open                0
high                0
low                 0
close               0
volume              0
trade_count         0
vwap                0
Next_Return         0
Target              0
SMA_10             36
ewm_10              0
price_to_ma10      36
SMA_20             76
ewm_20              0
price_to_ma20      76
SMA_50            196
ewm_50              0
price_to_ma50     196
SMA_100           396
ewm_100             0
price_to_ma100    396
SMA_200           796
ewm_200             0
price_to_ma200    796
ma_cross          796
BB_Mid             76
BB_Upper           76
BB_Lower           76
bb_pct             76
RSI                56
MACD                0
MACD_Signal         0
MACD_Hist           0
ret_1               4
log_ret             4
ret_lag1            8
ret_lag2           12
ret_lag3           16
mom_10             40
mom_20             80
mom_30            120
mom_50            200
mom_100           400
mom_200   

In [8]:
daily_data_feature.df.isna().sum()

symbol              0
timestamp           0
open                0
high                0
low                 0
close               0
volume              0
trade_count         0
vwap                0
Next_Return         0
Target              0
SMA_10             36
ewm_10              0
price_to_ma10      36
SMA_20             76
ewm_20              0
price_to_ma20      76
SMA_50            196
ewm_50              0
price_to_ma50     196
SMA_100           396
ewm_100             0
price_to_ma100    396
SMA_200           796
ewm_200             0
price_to_ma200    796
ma_cross          796
BB_Mid             76
BB_Upper           76
BB_Lower           76
bb_pct             76
RSI                56
MACD                0
MACD_Signal         0
MACD_Hist           0
ret_1               4
log_ret             4
ret_lag1            8
ret_lag2           12
ret_lag3           16
mom_10             40
mom_20             80
mom_30            120
mom_50            200
mom_100           400
mom_200   

In [9]:
# View one of the updated DataFrames with the new columns
weekly_data_feature.df.isna().sum()

symbol             0
timestamp          0
open               0
high               0
low                0
close              0
volume             0
trade_count        0
vwap               0
Next_Return        0
Target             0
SMA_5             16
ewm_5              0
price_to_ma5      16
SMA_10            36
ewm_10             0
price_to_ma10     36
SMA_20            76
ewm_20             0
price_to_ma20     76
SMA_30           116
ewm_30             0
price_to_ma30    116
ma_cross         116
BB_Mid            76
BB_Upper          76
BB_Lower          76
bb_pct            76
RSI               56
MACD               0
MACD_Signal        0
MACD_Hist          0
ret_1              4
log_ret            4
ret_lag1           8
ret_lag2          12
ret_lag3          16
mom_2              8
mom_4             16
mom_8             32
mom_12            48
mom_26           104
volatility_10     40
vol_ratio4        12
vol_ratio10       36
vol_ratio26      100
range_hl           0
gap          

<H5> Now we can safely discard these rows on the weekly data as it is no longer a significant data loss! <H5>

In [ ]:
#Replace infinities with NaN
hourly_data_feature.df = hourly_data_feature.df.replace([np.inf,-np.inf],np.nan)
daily_data_feature.df = daily_data_feature.df.replace([np.inf,-np.inf],np.nan)
weekly_data_feature.df = weekly_data_feature.df.replace([np.inf,-np.inf],np.nan)

#Drop NaN rows
hourly_data_feature.df.dropna(inplace = True)
daily_data_feature.df.dropna(inplace = True)
weekly_data_feature.df.dropna(inplace = True)

# View dataset again
hourly_data_feature.df.isna().sum()



symbol            0
timestamp         0
open              0
high              0
low               0
close             0
volume            0
trade_count       0
vwap              0
Next_Return       0
Target            0
SMA_10            0
ewm_10            0
price_to_ma10     0
SMA_20            0
ewm_20            0
price_to_ma20     0
SMA_50            0
ewm_50            0
price_to_ma50     0
SMA_100           0
ewm_100           0
price_to_ma100    0
SMA_200           0
ewm_200           0
price_to_ma200    0
ma_cross          0
BB_Mid            0
BB_Upper          0
BB_Lower          0
bb_pct            0
RSI               0
MACD              0
MACD_Signal       0
MACD_Hist         0
ret_1             0
log_ret           0
ret_lag1          0
ret_lag2          0
ret_lag3          0
mom_10            0
mom_20            0
mom_30            0
mom_50            0
mom_100           0
mom_200           0
volatility_20     0
vol_ratio10       0
vol_ratio20       0
vol_ratio50       0


In [ ]:
# View dataset again
daily_data_feature.df.isna().sum()

symbol            0
timestamp         0
open              0
high              0
low               0
close             0
volume            0
trade_count       0
vwap              0
Next_Return       0
Target            0
SMA_10            0
ewm_10            0
price_to_ma10     0
SMA_20            0
ewm_20            0
price_to_ma20     0
SMA_50            0
ewm_50            0
price_to_ma50     0
SMA_100           0
ewm_100           0
price_to_ma100    0
SMA_200           0
ewm_200           0
price_to_ma200    0
ma_cross          0
BB_Mid            0
BB_Upper          0
BB_Lower          0
bb_pct            0
RSI               0
MACD              0
MACD_Signal       0
MACD_Hist         0
ret_1             0
log_ret           0
ret_lag1          0
ret_lag2          0
ret_lag3          0
mom_10            0
mom_20            0
mom_30            0
mom_50            0
mom_100           0
mom_200           0
volatility_20     0
vol_ratio10       0
vol_ratio20       0
vol_ratio50       0


In [ ]:
# View dataset again
weekly_data_feature.df.isna().sum()

symbol           0
timestamp        0
open             0
high             0
low              0
close            0
volume           0
trade_count      0
vwap             0
Next_Return      0
Target           0
SMA_5            0
ewm_5            0
price_to_ma5     0
SMA_10           0
ewm_10           0
price_to_ma10    0
SMA_20           0
ewm_20           0
price_to_ma20    0
SMA_30           0
ewm_30           0
price_to_ma30    0
ma_cross         0
BB_Mid           0
BB_Upper         0
BB_Lower         0
bb_pct           0
RSI              0
MACD             0
MACD_Signal      0
MACD_Hist        0
ret_1            0
log_ret          0
ret_lag1         0
ret_lag2         0
ret_lag3         0
mom_2            0
mom_4            0
mom_8            0
mom_12           0
mom_26           0
volatility_10    0
vol_ratio4       0
vol_ratio10      0
vol_ratio26      0
range_hl         0
gap              0
atr_10           0
adx_10           0
mfi_10           0
cmf_10           0
dtype: int64

## 6. Data Preprocessing

At this point all three tables are ready to be preprocessed for ML algorithms